# VQPAINT painting generator (Kaggle / Colab)

Generates VQGAN+CLIP *paintings* with the repo's classic engine (`engine.py`) from a few thousand varied prompts and zips them.
Each painting is a 256x256 PNG plus its 16x16 VQGAN token grid; a later browser step builds a token bank from them.

**Kaggle:** Settings (right panel) -> Accelerator **GPU T4 x2** (or P100), **Internet: On**, then *Save Version -> Save & Run All*.
The output is `/kaggle/working/paintings.zip` (Output tab). Full steps, timings and how to resume after the 12 h limit:
`web/export/paintings/README.md` in the repo.

**Colab:** same cells; everything lives under `/content` (see comments) and you download `/content/paintings.zip` yourself.

In [ ]:
S = dict(
    COUNT = 1500,          # paintings (prompt indices 0..COUNT-1)
    ITERATIONS=150,      # optimisation steps per painting
    SIZE=256,            # px, multiple of 16 -> 16x16 tokens
    CUTN=32,
    SEED=0,
    TIME_BUDGET_H = 11,  # stop starting new paintings after this many hours (Kaggle kills sessions at 12 h); rerun with resume
    RESUME_FROM="/kaggle/input/*/paintings.zip",  # a previous version's output added as a data source (README); "" = ignore
)

import glob, os, shutil, subprocess, sys, time
ON_KAGGLE = os.path.isdir("/kaggle")
WORK = "/kaggle/working" if ON_KAGGLE else "/content"   # Kaggle saves this dir as the version's output; Colab: /content
TMP = "/kaggle/tmp" if ON_KAGGLE else "/content/tmp"    # not saved: repo clone, CLIP, taming, 1 GB checkpoint
REPO = f"{TMP}/VQPAINT"
VQC = f"{TMP}/01_vqgan_clip/VQGAN-CLIP"                 # == engine.py's default <repo>/../01_vqgan_clip/VQGAN-CLIP
OUT = f"{WORK}/paintings"
os.makedirs(TMP, exist_ok=True); os.makedirs(WORK, exist_ok=True)

def sh(cmd):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

print(S, "\nWORK", WORK, " TMP", TMP)

In [ ]:
# Setup: from the private asset dataset when attached (no internet needed), else clone/download online.
ASSETS = next(iter(glob.glob("/kaggle/input/vqpaint-assets*")), None)
if ASSETS:
    print("offline assets:", ASSETS)
    if not os.path.isdir(REPO): shutil.copytree(f"{ASSETS}/VQPAINT", REPO)
    os.makedirs(VQC, exist_ok=True)
    for d in ("CLIP", "taming-transformers"):
        if not os.path.isdir(f"{VQC}/{d}"): shutil.copytree(f"{ASSETS}/vqgan_clip/{d}", f"{VQC}/{d}")
    if not os.path.isdir(f"{VQC}/checkpoints"): shutil.copytree(f"{ASSETS}/checkpoints", f"{VQC}/checkpoints")
    os.makedirs(os.path.expanduser("~/.cache/clip"), exist_ok=True)
    if not os.path.exists(os.path.expanduser("~/.cache/clip/ViT-B-32.pt")): shutil.copy(f"{ASSETS}/clip_cache/ViT-B-32.pt", os.path.expanduser("~/.cache/clip/ViT-B-32.pt"))
    sh(f"{sys.executable} -m pip install -q --no-index --find-links {ASSETS}/wheels kornia kornia_rs ftfy regex tqdm omegaconf einops || {sys.executable} -m pip install -q kornia ftfy regex tqdm omegaconf einops")
else:
    if not os.path.isdir(REPO):
        sh(f"git clone -q --branch web-spikes --depth 1 https://github.com/NazarGol/VQPAINT.git {REPO}")
    os.makedirs(f"{VQC}/checkpoints", exist_ok=True)
    if not os.path.isdir(f"{VQC}/CLIP"):
        sh(f"git clone -q --depth 1 https://github.com/openai/CLIP.git {VQC}/CLIP")
    if not os.path.isdir(f"{VQC}/taming-transformers"):
        sh(f"git clone -q --depth 1 https://github.com/CompVis/taming-transformers.git {VQC}/taming-transformers")
    CK = f"{VQC}/checkpoints/vqgan_imagenet_f16_16384"
    if not os.path.exists(CK + ".yaml"):
        sh(f"curl -sSL 'https://heibox.uni-heidelberg.de/f/274fb24ed38341bfa753/?dl=1' -o {CK}.yaml")
    if not (os.path.exists(CK + ".ckpt") and os.path.getsize(CK + ".ckpt") > 900_000_000):
        sh(f"curl -sSL 'https://heibox.uni-heidelberg.de/f/867b05fc8c4841768640/?dl=1' -o {CK}.ckpt")
    sh(f"{sys.executable} -m pip install -q kornia ftfy regex tqdm omegaconf einops")
sh(f"{sys.executable} {REPO}/web/export/paintings/taming_patch.py {VQC}/taming-transformers")   # no pytorch-lightning needed
sh(f"ls -la {VQC} {VQC}/checkpoints")


In [ ]:
# Resume an earlier session: add that notebook version's output as a data source (Add Input -> Your Work);
# it appears as /kaggle/input/<slug>/paintings.zip, gets unpacked here, and --resume below skips finished indices.
zips = sorted(glob.glob(S["RESUME_FROM"])) if S["RESUME_FROM"] else []
if zips:
    print("resuming from", zips[-1])
    shutil.unpack_archive(zips[-1], WORK)
print(len(glob.glob(f"{OUT}/**/p*.png", recursive=True)), "paintings already in", OUT)

In [ ]:
# Generate: one gen_paintings.py process per GPU (shards), logs in WORK/gen_gpu*.log, a progress line every 2 minutes.
import torch
n_gpu = max(1, torch.cuda.device_count())
print(n_gpu, "GPU(s):", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
procs = []
for k in range(n_gpu):
    out_k = OUT if n_gpu == 1 else f"{OUT}/gpu{k}"
    cmd = [sys.executable, f"{REPO}/web/export/paintings/gen_paintings.py", "--out", out_k,
           "--count", str(S["COUNT"]), "--iterations", str(S["ITERATIONS"]), "--size", str(S["SIZE"]),
           "--cutn", str(S["CUTN"]), "--seed", str(S["SEED"]), "--resume", "--shard", f"{k}/{n_gpu}",
           "--device", f"cuda:{k}" if torch.cuda.is_available() else "cpu",
           "--time-budget-h", str(S["TIME_BUDGET_H"]), "--vqgan-clip-dir", VQC]
    log = open(f"{WORK}/gen_gpu{k}.log", "a")
    procs.append((subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT), log))
t0 = time.time()
while any(p.poll() is None for p, _ in procs):
    time.sleep(120)
    n_png = len(glob.glob(f"{OUT}/**/p*.png", recursive=True))
    tails = []
    for k in range(n_gpu):
        lines = open(f"{WORK}/gen_gpu{k}.log", errors="replace").read().strip().splitlines()
        tails.append(lines[-1][:80] if lines else "")
    print(f"{(time.time() - t0) / 3600:5.2f} h  {n_png} paintings | " + " | ".join(tails), flush=True)
for k, (p, log) in enumerate(procs):
    log.close()
    print(f"gpu{k}: exit code {p.returncode}  (see {WORK}/gen_gpu{k}.log)")

In [ ]:
# Merge shard token files (T4 x2) and zip everything for download / for resuming in the next session.
if n_gpu > 1:
    sh(f"{sys.executable} {REPO}/web/export/paintings/gen_paintings.py --merge --out {OUT}")
n_png = len(glob.glob(f"{OUT}/**/p*.png", recursive=True))
zip_path = shutil.make_archive(f"{WORK}/paintings", "zip", root_dir=WORK, base_dir="paintings")
print(f"{n_png} paintings -> {zip_path} ({os.path.getsize(zip_path) / 2**20:.0f} MiB)")
print("Kaggle: download it from this version's Output tab once the run finishes. Colab: from the Files panel or files.download(zip_path).")